# 03 - Model Architecture
## PhishSense: Multimodal Fusion Model

This notebook covers:
1. **NLP Branch**: DistilBERT + BiLSTM + Attention for URL text processing
2. **Numerical Branch**: MLP and CapsNet for engineered features
3. **Fusion Model**: Hierarchical concatenation of both branches
4. **XGBoost Classifier**: Final decision-maker on fused features
5. Architecture visualization and parameter counts

```
                    ┌─────────────────────────────────────┐
                    │           Input: URL String          │
                    └──────────┬──────────┬───────────────┘
                               │          │
                    ┌──────────▼──┐  ┌────▼──────────────┐
                    │  NLP Branch │  │ Numerical Branch   │
                    │             │  │                    │
                    │ DistilBERT  │  │ Feature Extraction │
                    │     ↓       │  │ (24 features)      │
                    │  BiLSTM     │  │       ↓            │
                    │     ↓       │  │  MLP / CapsNet     │
                    │ Attention   │  │                    │
                    │     ↓       │  │       ↓            │
                    │  FC (128d)  │  │   FC (64d)         │
                    └──────┬──────┘  └────┬───────────────┘
                           │              │
                    ┌──────▼──────────────▼───────────────┐
                    │   Concatenation (192d fused vector) │
                    └──────────────┬──────────────────────┘
                                   │
                    ┌──────────────▼──────────────────────┐
                    │       XGBoost Classifier            │
                    │    (Binary: phishing / benign)      │
                    └─────────────────────────────────────┘
```

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3.1 NLP Branch: DistilBERT + BiLSTM + Attention

This branch processes the URL as a text string:
- **DistilBERT**: Pre-trained language model extracts contextual token embeddings (768d)
- **BiLSTM**: Captures sequential dependencies in both directions
- **Attention**: Learns to focus on suspicious tokens/subwords
- **FC**: Projects to 128-dimensional output vector

In [ ]:
class AttentionLayer(nn.Module):
    """Attention mechanism to weight important tokens in URL."""

    def __init__(self, hidden_size: int):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, lstm_output: torch.Tensor) -> torch.Tensor:
        # lstm_output shape: (batch, seq_len, hidden*2)
        weights = torch.softmax(self.attention(lstm_output), dim=1)  # (batch, seq_len, 1)
        return torch.sum(weights * lstm_output, dim=1)  # (batch, hidden*2)


class NLPBranch(nn.Module):
    """
    DistilBERT -> BiLSTM -> Attention -> FC
    Processes URL string to extract semantic features.
    """

    def __init__(
        self,
        output_dim: int = 128,
        lstm_hidden: int = 256,
        lstm_layers: int = 2,
        dropout: float = 0.3,
        freeze_bert: bool = True,
    ):
        super().__init__()

        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze_bert:
            for param in self.distilbert.parameters():
                param.requires_grad = False

        bert_hidden = self.distilbert.config.hidden_size  # 768

        self.bilstm = nn.LSTM(
            input_size=bert_hidden,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0,
        )

        self.attention = AttentionLayer(lstm_hidden * 2)  # *2 for bidirectional
        self.fc = nn.Linear(lstm_hidden * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        # DistilBERT encoding
        bert_output = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_output.last_hidden_state  # (batch, seq_len, 768)

        # BiLSTM
        lstm_output, _ = self.bilstm(sequence_output)  # (batch, seq_len, hidden*2)

        # Attention pooling
        attended = self.attention(lstm_output)  # (batch, hidden*2)

        # Final projection
        output = self.fc(self.dropout(attended))  # (batch, output_dim)
        return output


class URLTokenizer:
    """Tokenizes URLs for DistilBERT input."""

    def __init__(self, max_length: int = 128):
        self.tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_length = max_length

    def tokenize(self, urls: list[str]) -> dict:
        return self.tokenizer(
            urls,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )


print("NLP Branch defined successfully.")

In [ ]:
# Test NLP Branch
nlp_branch = NLPBranch(output_dim=128, freeze_bert=True)

tokenizer = URLTokenizer()
test_urls = ["https://www.google.com", "http://paypa1-secure.tk/login"]
tokens = tokenizer.tokenize(test_urls)

with torch.no_grad():
    nlp_output = nlp_branch(tokens["input_ids"], tokens["attention_mask"])

print(f"Input token shape:  {tokens['input_ids'].shape}")
print(f"NLP Branch output:  {nlp_output.shape}")
print(f"Expected:           (2, 128)")

## 3.2 Numerical Branch: MLP

Processes the 24 engineered features through a multi-layer perceptron with batch normalization and dropout.

In [ ]:
class MLPBranch(nn.Module):
    """Multi-Layer Perceptron for processing numerical URL features."""

    def __init__(
        self,
        input_dim: int = 24,
        hidden_dims: list[int] | None = None,
        output_dim: int = 64,
        dropout: float = 0.3,
    ):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = [128, 64]

        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


# Test MLP Branch
mlp_branch = MLPBranch(input_dim=24, output_dim=64)
dummy_features = torch.randn(2, 24)

with torch.no_grad():
    mlp_output = mlp_branch(dummy_features)

print(f"MLP input shape:   {dummy_features.shape}")
print(f"MLP output shape:  {mlp_output.shape}")
print(f"Expected:          (2, 64)")
print(f"\nMLP Architecture:\n{mlp_branch}")

## 3.3 Alternative Numerical Branch: Capsule Neural Network (CapsNet)

CapsNet preserves hierarchical structural information from engineered URL features without losing spatial properties.

In [ ]:
class PrimaryCapsule(nn.Module):
    """Primary capsule layer for CapsNet."""

    def __init__(self, in_channels: int, out_channels: int, capsule_dim: int):
        super().__init__()
        self.capsule_dim = capsule_dim
        self.conv = nn.Linear(in_channels, out_channels * capsule_dim)
        self.out_channels = out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self.conv(x)
        return output.view(-1, self.out_channels, self.capsule_dim)


def squash(tensor: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Squash activation function for capsule networks."""
    squared_norm = (tensor**2).sum(dim=dim, keepdim=True)
    scale = squared_norm / (1 + squared_norm)
    return scale * tensor / (torch.sqrt(squared_norm) + 1e-8)


class CapsNetBranch(nn.Module):
    """
    Capsule Neural Network for preserving hierarchical structural
    information from engineered URL features.
    """

    def __init__(
        self,
        input_dim: int = 24,
        primary_caps: int = 8,
        capsule_dim: int = 8,
        output_dim: int = 64,
    ):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.primary_capsules = PrimaryCapsule(128, primary_caps, capsule_dim)
        self.fc_out = nn.Linear(primary_caps * capsule_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.bn1(self.fc1(x)))
        capsules = self.primary_capsules(x)
        capsules = squash(capsules)
        flattened = capsules.view(capsules.size(0), -1)
        return self.fc_out(flattened)


# Test CapsNet Branch
capsnet_branch = CapsNetBranch(input_dim=24, output_dim=64)
dummy_features = torch.randn(2, 24)

with torch.no_grad():
    caps_output = capsnet_branch(dummy_features)

print(f"CapsNet input shape:   {dummy_features.shape}")
print(f"CapsNet output shape:  {caps_output.shape}")
print(f"Expected:              (2, 64)")
print(f"\nCapsNet Architecture:\n{capsnet_branch}")

## 3.4 Fusion Model: Combining Both Branches

The fusion model concatenates outputs from both branches into a single 192-dimensional feature vector.

In [ ]:
import xgboost as xgb
import numpy as np


class PhishSenseFusionModel(nn.Module):
    """
    Hierarchical fusion model that combines:
    1. NLP Branch (DistilBERT + BiLSTM + Attention) for URL text
    2. Numerical Branch (MLP) for engineered features
    """

    def __init__(
        self,
        num_features: int = 24,
        nlp_output_dim: int = 128,
        numerical_output_dim: int = 64,
        freeze_bert: bool = True,
    ):
        super().__init__()

        self.nlp_branch = NLPBranch(
            output_dim=nlp_output_dim,
            freeze_bert=freeze_bert,
        )
        self.numerical_branch = MLPBranch(
            input_dim=num_features,
            output_dim=numerical_output_dim,
        )

        self.fusion_dim = nlp_output_dim + numerical_output_dim

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        numerical_features: torch.Tensor,
    ) -> torch.Tensor:
        """Extract fused feature vector from both branches."""
        nlp_output = self.nlp_branch(input_ids, attention_mask)
        numerical_output = self.numerical_branch(numerical_features)

        # Hierarchical fusion via concatenation
        fused = torch.cat([nlp_output, numerical_output], dim=1)
        return fused


class PhishSenseClassifier:
    """
    Complete PhishSense classifier:
    Neural network feature extractor + XGBoost final classifier.
    """

    def __init__(self, fusion_model: PhishSenseFusionModel):
        self.fusion_model = fusion_model
        self.xgb_classifier = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            objective="binary:logistic",
            eval_metric="logloss",
            use_label_encoder=False,
        )

    def extract_features(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        numerical_features: torch.Tensor,
    ) -> np.ndarray:
        """Extract fused features using neural network."""
        self.fusion_model.eval()
        with torch.no_grad():
            fused = self.fusion_model(input_ids, attention_mask, numerical_features)
        return fused.cpu().numpy()

    def fit(self, input_ids, attention_mask, numerical_features, labels):
        """Train XGBoost on fused features."""
        features = self.extract_features(input_ids, attention_mask, numerical_features)
        self.xgb_classifier.fit(features, labels)

    def predict(self, input_ids, attention_mask, numerical_features) -> np.ndarray:
        """Predict phishing probability."""
        features = self.extract_features(input_ids, attention_mask, numerical_features)
        return self.xgb_classifier.predict_proba(features)[:, 1]


print("PhishSenseFusionModel and PhishSenseClassifier defined.")

## 3.5 Test Full Fusion Pipeline & Parameter Summary

In [ ]:
# Instantiate full fusion model
fusion_model = PhishSenseFusionModel(num_features=24, freeze_bert=True)

# Test forward pass with dummy data
dummy_input_ids = torch.randint(0, 30522, (2, 32))
dummy_attention_mask = torch.ones(2, 32, dtype=torch.long)
dummy_numerical = torch.randn(2, 24)

with torch.no_grad():
    fused_output = fusion_model(dummy_input_ids, dummy_attention_mask, dummy_numerical)

print(f"Fused output shape: {fused_output.shape}")
print(f"Fusion dimension:   {fusion_model.fusion_dim}")
print(f"Expected:           (2, 192)")

# Parameter count
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen

total, trainable, frozen = count_parameters(fusion_model)
print(f"\n{'='*50}")
print(f"PARAMETER SUMMARY")
print(f"{'='*50}")
print(f"Total parameters:     {total:>12,}")
print(f"Trainable parameters: {trainable:>12,}")
print(f"Frozen parameters:    {frozen:>12,}")
print(f"{'='*50}")

# Per-component breakdown
nlp_total, nlp_train, _ = count_parameters(fusion_model.nlp_branch)
num_total, num_train, _ = count_parameters(fusion_model.numerical_branch)
print(f"\nNLP Branch:       {nlp_total:>10,} total, {nlp_train:>10,} trainable")
print(f"Numerical Branch: {num_total:>10,} total, {num_train:>10,} trainable")